# TripoSR：单张图片生成 3D 模型

依次运行以下单元格。先在 Colab 的“运行时 → 更改运行时类型”中选择 GPU，并在下载单元格中填写你自己的私有或临时项目压缩包地址。

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), '当前没有分配到 GPU，请在运行时设置中选择 GPU 后重新运行。'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
print('无需挂载 Google Drive；下载地址由使用者在下一单元格中自行填写。')

In [ ]:
from pathlib import Path
zip_path = Path('/content/Code_Group12.zip')
PROJECT_ZIP_URL = ''  # Paste your own private or temporary download URL here.
assert PROJECT_ZIP_URL, '请先在 PROJECT_ZIP_URL 中填写你自己的项目压缩包下载地址'
!curl -L "{PROJECT_ZIP_URL}" -o "{zip_path}"
assert zip_path.exists() and zip_path.stat().st_size > 1_000_000_000, '项目下载失败或文件不完整'
!rm -rf /content/Code_Group12
!mkdir -p /content/Code_Group12
!unzip -q "{zip_path}" -d /content/Code_Group12
%cd /content/Code_Group12
print('项目下载并解压完成')

In [ ]:
!pip -q install 'setuptools<82' wheel
!pip -q install omegaconf==2.3.0 Pillow einops==0.7.0 transformers==4.35.0 tokenizers==0.14.1 huggingface-hub==0.17.3 trimesh==4.0.5 'rembg[cpu]' 'imageio[ffmpeg]' xatlas==0.0.9 moderngl==5.10.0
!pip -q install git+https://github.com/tatsy/torchmcubes.git
print('依赖安装完成。若 Colab 提示需要重启运行时，请重启后从本单元格之后继续。')

## 选择输入图片并生成 OBJ

In [ ]:
from google.colab import files
uploaded = files.upload()
assert uploaded, '请上传一张 PNG 或 JPG 图片'
input_name = next(iter(uploaded))
print('输入图片：', input_name)

In [ ]:
!rm -rf /content/triposr_output
!python run.py "{input_name}" --output-dir /content/triposr_output --mc-resolution 256
print('生成完成。')

In [ ]:
from pathlib import Path
from google.colab import files
results = list(Path('/content/triposr_output').rglob('mesh.obj'))
assert results, '没有找到输出模型，请检查上一单元格的错误信息。'
files.download(str(results[0]))